# 02 — Field-conditioned synthetic geology and exact AVO generation

| Item | Definition |
|---|---|
| **Scientific purpose** | Turn the calibrated Stage-01 structural/elastic background into geologically diverse, physics-generated AVO realizations. |
| **Inputs** | Stage-01 Vp, Vs, density, DELTA/P(sand), porosity, RGT, reservoir mask, stratigraphic fraction, blend weights, and the fitted reservoir elastic model. |
| **Outputs** | Complete realization packages containing geology, brine and substituted elastic properties, dense-angle exact PP AVO, three angle stacks, PWD dip, masks, coordinates, and provenance. |
| **Data availability** | Algorithms and configuration schemas are public. Licensed S01 inputs and generated arrays remain local. |
| **Local/private-data requirements** | Configure `work_data_root` and `private_artifact_root` in ignored `configs/paths.yaml`. No synthetic toy fallback is used when Stage-01 artifacts are absent. |
| **Software requirements** | `pip install -e ".[field,ml,notebooks]"`; Madagascar is optional and used only for historical production-path cross-checking. |
| **Approximate runtime** | Minutes per realization on CPU; the configured 200-realization production family is an offline generation job. |
| **Pipeline position** | Consumes Notebook 01; produces the full realizations consumed by Notebook 03. |

The main forward operator is the **exact PP Zoeppritz solution** followed by wavelet convolution and angle-domain mute/taper. Shuey/Aki–Richards intercept and gradient are compact diagnostics and later model features—not substitutes for the generation physics.

In [1]:
from pathlib import Path

def find_repository_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "sage_avo").exists():
            return candidate
    raise RuntimeError("Run this notebook from the installed SAGE-AVO repository.")

ROOT = find_repository_root()

import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sage_avo.config import load_config, seed_everything
from sage_avo.experiments import (
    forward_config_from_mapping,
    generate_stage02_dataset,
    load_stage01_background,
    load_stage02_manifest,
)
from sage_avo.forward import forward_avo_dense, forward_avo_madagascar, madagascar_availability

workflow = load_config(ROOT / "configs" / "synthetic_s01.yaml")
paths_file = ROOT / "configs" / "paths.yaml"
if not paths_file.exists():
    raise FileNotFoundError(
        "Create ignored configs/paths.yaml from configs/paths.example.yaml and point it "
        "to the licensed Stage-01 artifacts and private output root."
    )
paths = load_config(paths_file)
seed_everything(int(workflow["stage"]["seed"]))

private_root = Path(paths["private_artifact_root"])
realization_dir = private_root / "stage_artifacts" / "stage02" / "realizations"
figure_dir = private_root / "figures" / "stage02"
figure_dir.mkdir(parents=True, exist_ok=True)

## 1. Stage-01 contract and conventions

All channels share the Stage-01 time/CDP grid. `elastic_background` and `elastic_blend_weight` have shape `[3, time, trace]`; the other image channels have shape `[time, trace]`.

The canonical convention is

\[
\mathrm{DELTA}=\text{shaliness},\qquad P(\mathrm{sand})=1-\mathrm{DELTA}.
\]

Historical development code sometimes assigned the sand-probability field directly to `DELTA`. The production interface converts explicitly and never propagates that reversed convention. The field-calibrated probability range also makes a 0.5 threshold inappropriate here; the configured 0.30 threshold is supported by the Stage-01 reservoir distribution and is recorded in every realization manifest.

In [2]:
stage01, reservoir_model, source_hashes = load_stage01_background(
    paths["work_data_root"],
    workflow["inputs"]["dataset_id"],
    workflow["inputs"]["structure_version"],
)
contract = pd.DataFrame(
    [{"channel": name, "shape": value.shape, "dtype": value.dtype} for name, value in stage01.items()]
)
display(contract)
print(f"Hashed source artifacts: {len(source_hashes)}")
print(
    "Reservoir P(sand) range:",
    np.nanmin(stage01["sand_probability"][stage01["reservoir_mask"].astype(bool)]),
    np.nanmax(stage01["sand_probability"][stage01["reservoir_mask"].astype(bool)]),
)

,channel,shape,dtype
0,sand_probability,"(301, 160)",float32
1,porosity,"(301, 160)",float32
2,elastic_background,"(3, 301, 160)",float32
3,elastic_blend_weight,"(3, 301, 160)",float32
4,reservoir_mask,"(301, 160)",uint8
5,time_ms,"(301,)",float32
6,cdp,"(160,)",int32
7,rgt,"(301, 160)",float32
8,strat_fraction,"(301, 160)",float32
9,horizon_top_ms,"(160,)",float64


Hashed source artifacts: 12
Reservoir P(sand) range: 0.13369906 0.48170447


## 2. Deterministic geological realization

A realization ID is also its random seed. One coherent deformation field is applied to every Stage-01 channel, so horizons, RGT, facies, porosity, masks, and elastic background remain registered. The deformation combines smooth folds with optional finite-length fault displacement. Correlated Gaussian fields then perturb P(sand) and porosity inside the warped reservoir.

The trained Stage-01 random-forest relationship maps `[DELTA, porosity, stratigraphic fraction]` to reservoir Vp/Vs/density. Warped regional elastic background is preserved outside the reservoir and blended only across the saved transition weights; this prevents the block artifacts produced by assigning a constant exterior.

In [3]:
print("Configured realizations:", workflow["stage"]["realization_count"])
print("Geological deformation parameters:", {
    key: value for key, value in workflow["geology"].items()
    if "fold" in key or "fault" in key
})
print("Sand facies P(sand) threshold:", workflow["geology"]["sand_facies_probability_threshold"])
print("Fluid substitution:", workflow["fluid_substitution"])

Configured realizations: 200
Geological deformation parameters: {'fold_probability': 0.8, 'fold_amplitude_samples': [10.0, 30.0], 'fold_cycles': [0.3, 3.0], 'secondary_fold_amplitude_samples': [2.0, 8.0], 'secondary_fold_cycles': [2.0, 4.0], 'maximum_faults': 7, 'fault_throw_samples': [-40.0, 40.0], 'fault_dip_samples_per_trace': [-0.5, 0.5]}
Sand facies P(sand) threshold: 0.3
Fluid substitution: {'enabled': True, 'plume_count': [1, 2], 'plume_lateral_radius_samples': [15.0, 40.0], 'plume_vertical_radius_samples': [5.0, 15.0], 'minimum_sand_thickness_samples': 7, 'co2_saturation': [0.3, 0.8], 'critical_porosity': 0.36, 'coordination_factor': 2.8, 'quartz_bulk_modulus_gpa': 39.0, 'clay_bulk_modulus_gpa': 21.0, 'quartz_shear_modulus_gpa': 45.0, 'clay_shear_modulus_gpa': 6.85, 'quartz_density_g_cc': 2.65, 'clay_density_g_cc': 2.6, 'overburden_density_kg_m3': 1600.0, 'gravity_m_s2': 9.8, 'depth_origin_m': 2000.0, 'depth_increment_m': 4.0, 'brine_bulk_modulus_gpa': 2.2, 'co2_bulk_modulus_gp

## 3. CO₂ scenario and fluid substitution

CO₂ saturation is introduced only in connected, sufficiently thick reservoir sand. The implementation follows the historical field-conditioned rock-physics path: Hertz–Mindlin dry-frame moduli, Brie mixing for the brine/CO₂ fluid modulus, then Gassmann saturation. Density follows volumetric fluid replacement. Both the brine baseline and substituted elastic cubes are saved, making the physical change auditable.

## 4. Exact dense-angle forward response

For each elastic interface and each configured angle, the production operator solves the exact isotropic PP Zoeppritz system. A Ricker wavelet is convolved along time, then the historical front mute/taper is applied. Dense responses retain all 43 angles from 3° through 45°.

Angle bands are centralized in configuration. Historical products used overlapping endpoints (`3–17`, `17–31`, `31–45`). New products use non-overlapping integer-angle bands (`3–17`, `18–31`, `32–45`). Both definitions are written to the manifest; old products are not silently relabeled.

In [4]:
forward_definition = forward_config_from_mapping(workflow)
display(
    pd.DataFrame(
        [{"band": b.name, "minimum_deg": b.minimum_degrees, "maximum_deg": b.maximum_degrees}
         for b in forward_definition.bands]
    )
)
print("Legacy bands:", workflow["forward"]["legacy_bands"])
print("Dense angles:", forward_definition.angles_degrees)

,band,minimum_deg,maximum_deg
0,near,3.0,17.0
1,mid,18.0,31.0
2,far,32.0,45.0


Legacy bands: {'near': [3.0, 17.0], 'mid': [17.0, 31.0], 'far': [31.0, 45.0]}
Dense angles: (3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0)


## 5. Generate the realization family

The default call creates the complete configured family. `SAGE_AVO_STAGE02_LIMIT` is an explicit operator-validation control for local development; a limited manifest is labeled `operator_validation_subset` and must not be presented as a completed corpus. `SAGE_AVO_REUSE_STAGE02=1` reopens an existing immutable local result.

In [ ]:
limit_text = os.getenv("SAGE_AVO_STAGE02_LIMIT", "").strip()
realization_limit = int(limit_text) if limit_text else None
reuse = os.getenv("SAGE_AVO_REUSE_STAGE02", "0") == "1"
manifest_path = realization_dir / "manifest.json"

if reuse and manifest_path.exists():
    manifest = load_stage02_manifest(manifest_path)
else:
    manifest = generate_stage02_dataset(
        config=workflow,
        paths=paths,
        output_directory=realization_dir,
        realization_limit=realization_limit,
    )

display(pd.Series({key: manifest[key] for key in (
    "status", "requested_realizations", "generated_realizations", "exact_forward_operator"
)}).to_frame("value"))

## 6. Deterministic realization QC

The representative realization is the smallest generated ID—a documented rule independent of visual appearance. The panels verify channel registration, the DELTA/P(sand) complement, plume support, exact near/mid/far response, and recalculated PWD dip. RGT is coherently warped from Stage 01; the historical synthetic workflow recalculated dip rather than reintegrating RGT.

In [ ]:
representative_id = min(manifest["realization_ids"])
representative_path = realization_dir / f"realization_{representative_id:07d}.npz"
with np.load(representative_path, allow_pickle=False) as archive:
    realization = {name: archive[name] for name in archive.files}

panels = [
    (realization["sand_probability"], "P(sand)", "viridis"),
    (realization["delta"], "DELTA (shaliness)", "viridis_r"),
    (realization["porosity"], "Porosity", "viridis"),
    (realization["co2_saturation"], "CO₂ saturation", "magma"),
    (realization["elastic"][0], "Vp", "viridis"),
    (realization["elastic"][1], "Vs", "viridis"),
    (realization["elastic"][2], "Density", "viridis"),
    (realization["rgt"], "Warped RGT", "turbo"),
    (realization["avo"][0], "Near AVO", "gray"),
    (realization["avo"][1], "Mid AVO", "gray"),
    (realization["avo"][2], "Far AVO", "gray"),
    (realization["dip_pwd"], "Recalculated PWD dip", "coolwarm"),
]
fig, axes = plt.subplots(3, 4, figsize=(16, 10), constrained_layout=True)
for axis, (array, title, cmap) in zip(axes.flat, panels):
    image = axis.imshow(array, aspect="auto", cmap=cmap)
    axis.set_title(title)
    axis.set_xlabel("Trace")
    axis.set_ylabel("Time sample")
    fig.colorbar(image, ax=axis, shrink=0.72)
for axis in axes[:2].flat:
    top_sample = np.interp(realization["horizon_top_ms"], realization["time_ms"], np.arange(realization["time_ms"].size))
    base_sample = np.interp(realization["horizon_base_ms"], realization["time_ms"], np.arange(realization["time_ms"].size))
    axis.plot(top_sample, color="white", linewidth=0.8, label="warped T6")
    axis.plot(base_sample, color="black", linewidth=0.8, label="warped T7")
fig.suptitle(f"Stage-02 field-conditioned realization {representative_id}")
qc_path = figure_dir / "stage02_representative_realization.png"
fig.savefig(qc_path, dpi=300, bbox_inches="tight")
plt.show()

print("max |DELTA + P(sand) - 1| =", np.max(np.abs(
    realization["delta"] + realization["sand_probability"] - 1.0
)))
print("private figure:", qc_path)

## 7. Historical Madagascar production-path cross-check

If Madagascar is installed, the same elastic crop is passed through `sfzoeppritz2 → sftransp → sfricker1 → sftransp`. Correlation is the principal diagnostic because the historical `sfricker1` normalization differs from the NumPy wavelet normalization. This is a verification path; it does not change the manifest’s chosen local backend.

In [ ]:
availability = madagascar_availability()
print(availability)
if availability.available:
    crop = realization["elastic"][:, 80:130, 20:120]
    numpy_forward = forward_avo_dense(*crop, config=forward_definition)
    rsf_forward = forward_avo_madagascar(*crop, config=forward_definition)
    comparisons = []
    for band, numpy_stack, rsf_stack in zip(
        numpy_forward.band_names, numpy_forward.stacks, rsf_forward.stacks
    ):
        comparisons.append({
            "band": band,
            "correlation": np.corrcoef(numpy_stack.ravel(), rsf_stack.ravel())[0, 1],
            "standard_deviation_ratio_rsf_to_numpy": rsf_stack.std() / numpy_stack.std(),
        })
    display(pd.DataFrame(comparisons))
else:
    print("Madagascar cross-check skipped; exact NumPy Zoeppritz remains the configured operator.")

## 8. Saved-channel manifest

In [ ]:
channel_table = pd.DataFrame(
    [{"channel": name, **definition} for name, definition in manifest["channels"].items()]
)
display(channel_table)

## Stage outputs

| artifact | shape/type | scientific meaning | consumed by |
|---|---|---|---|
| `realization_XXXXXXX.npz` | dense 43-angle AVO; 3-band AVO; 3-channel elastic; geological/structural masks | One deterministic field-conditioned geological and exact-physics experiment | Notebook 03 |
| per-realization JSON | provenance + deformation/fluid parameters + QC | Reproducibility record | Notebooks 03 and 05 |
| `manifest.json` | channel schema, hashes, IDs, conventions | Immutable Stage-02 dataset contract | Notebook 03 |

## Scientific checks

- Shared deformation keeps geology, RGT, masks, and elastic fields registered.
- `DELTA + P(sand) = 1` is checked numerically.
- Elastic and AVO channels are finite and physically bounded in each saved QC record.
- CO₂ substitution is confined to connected reservoir sand; brine and substituted elastic cubes are both retained.
- Exact dense-angle Zoeppritz is the primary operator; compact P/G approximations are not used to generate training observations.
- Current and legacy angle bands remain separately identified.
- Optional Madagascar correlation checks the historical production route.

## Next stage

Notebook 03 consumes the immutable realization IDs and complete saved channels. It splits at the **realization level**, constructs the disclosed truth-derived low-frequency elastic prior, and extracts traceable multiscale patches without leakage.